# Who pays when the rupee falls?

Save inputs.json in the working directory. Run the Python cells to reproduce the results CSV. The chart cell uses Matplotlib. This is a fixed-input comparison.

In [ ]:
import csv
import json
from pathlib import Path

a = json.loads(Path("inputs.json").read_text())
for name, url in a["sources"].items():
    print(f"{name} | {url}")


In [ ]:
barrels = a["import_volume_mt"] * 1e6 * a["barrels_per_tonne"]
rupees_per_barrel = a["baseline_oil_usd"] * (
    a["stress_inr_per_usd"] - a["baseline_inr_per_usd"]
)
crude_delta = rupees_per_barrel / a["litres_per_barrel"]
rupee_import_delta_crore = barrels * rupees_per_barrel / 1e7
dollar_import_delta = 0  # The dollar oil price and import volume are held fixed.
rows = []
for share in a["pass_through"]:
    pump_delta = crude_delta * share * (1 + a["delhi_vat_rate"])
    rows.append((
        round(100 * share), pump_delta,
        a["delhi_petrol_inr_per_litre"] + pump_delta,
        pump_delta * a["household_litres_per_month"],
    ))

assert rows[0][1:] == (0, a["delhi_petrol_inr_per_litre"], 0)
assert all(b[1] >= a[1] for a, b in zip(rows, rows[1:]))
print(f"Crude cost change per litre: ₹{crude_delta:.2f}")
print(f"Fixed-volume rupee exposure: ₹{rupee_import_delta_crore:,.0f} crore")

with Path("results.csv").open("w", newline="") as handle:
    writer = csv.writer(handle)
    writer.writerow((
        "pass_through_pct", "pump_change_inr_per_litre",
        "pump_price_inr_per_litre", "household_change_inr_per_month",
    ))
    writer.writerows((share, *(f"{x:.2f}" for x in amounts)) for share, *amounts in rows)


The Delhi retail baseline comes from Indian Oil. Delhi VAT applies to the modeled cost change. The import comparison fixes six-month volume and uses an approximate tonne-to-barrel conversion. Real invoices vary with the grade mix, freight, discounts and hedges.

In [ ]:
import matplotlib.pyplot as plt

paper, ink, accent, muted = "#FAF8F4", "#15171A", "#B42318", "#555A61"
fig, ax = plt.subplots(figsize=(12, 6.3), dpi=100)
fig.patch.set_facecolor(paper)
ax.set_facecolor(paper)
fig.subplots_adjust(left=0.13, right=0.9, top=0.73, bottom=0.2)
shares, monthly = [f"{r[0]}%" for r in rows], [r[3] for r in rows]
ax.barh(shares, monthly, color=accent, height=0.6, zorder=3)
ax.set_xlim(0, 210)
ax.set_xticks([0, 45, 90, 135, 180], ["₹0", "₹45", "₹90", "₹135", "₹180"])
ax.invert_yaxis()
ax.grid(axis="x", color="#D8D4CC", linewidth=0.8)
ax.tick_params(length=0, labelsize=12, colors=muted, pad=8)
ax.set_ylabel("Cost passed through", labelpad=12, color=muted, fontsize=11)
for spine in ax.spines.values():
    spine.set_visible(False)
for i, v in enumerate(monthly):
    ax.text(v + 4, i, f"₹{v:.0f}", va="center", color=ink, fontsize=13)
fig.text(0.075, 0.92, "THE KAKISTOCRAT  /  INDIA POLICY RESEARCH", color=accent, fontsize=11, weight="bold")
fig.text(0.075, 0.85, "Who pays when the rupee falls?", color=ink, fontsize=29, weight="bold")
fig.text(0.075, 0.786, "Change in a Delhi household’s monthly petrol bill at 40 litres", color=muted, fontsize=15)
fig.text(0.075, 0.095, "Scenario: ₹94 to ₹100 per dollar, oil at $100 a barrel, excise held fixed. Source: Indian Oil and Delhi Budget.", color=muted, fontsize=10)
fig.savefig("pass-through.png", facecolor=paper)
plt.close(fig)
